In [1]:

# Lab 8: Playing Atari Games using Deep Q-Network (DQN)

"""
This lab teaches how to train a Deep Q-Network (DQN) to play Atari games (e.g., Breakout)
using Gymnasium (Atari environments), TensorFlow/Keras, and OpenCV.
"""

# Step 1: Install Required Packages
# !pip install gymnasium[atari,accept-rom-license] opencv-python

import gymnasium as gym
import numpy as np
import random
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers
from collections import deque
import cv2
import matplotlib.pyplot as plt

# Step 2: Frame Preprocessing
def preprocess_frame(frame):
    gray = cv2.cvtColor(frame, cv2.COLOR_RGB2GRAY)
    resized = cv2.resize(gray, (84, 84))
    normalized = resized / 255.0
    return normalized

# Step 3: Stack Frames
class FrameStack:
    def __init__(self, stack_size=4):
        self.stack_size = stack_size
        self.frames = deque(maxlen=stack_size)

    def reset(self, state):
        processed = preprocess_frame(state)
        self.frames = deque([processed] * self.stack_size, maxlen=self.stack_size)
        return np.stack(self.frames, axis=2)

    def step(self, state):
        processed = preprocess_frame(state)
        self.frames.append(processed)
        return np.stack(self.frames, axis=2)

# Step 4: Build DQN Model
def build_dqn(input_shape, n_actions):
    inputs = layers.Input(shape=input_shape)
    x = layers.Conv2D(32, (8, 8), strides=4, activation='relu')(inputs)
    x = layers.Conv2D(64, (4, 4), strides=2, activation='relu')(x)
    x = layers.Conv2D(64, (3, 3), strides=1, activation='relu')(x)
    x = layers.Flatten()(x)
    x = layers.Dense(512, activation='relu')(x)
    outputs = layers.Dense(n_actions, activation='linear')(x)
    model = models.Model(inputs=inputs, outputs=outputs)
    model.compile(optimizer=optimizers.Adam(learning_rate=0.00025), loss='huber')
    return model

# Step 5: Initialize Environment
env = gym.make("ALE/Breakout-v5", render_mode=None)
n_actions = env.action_space.n
frame_stack = FrameStack()

main_model = build_dqn((84, 84, 4), n_actions)
target_model = build_dqn((84, 84, 4), n_actions)
target_model.set_weights(main_model.get_weights())

memory = deque(maxlen=100_000)

def remember(s, a, r, s_, done):
    memory.append((s, a, r, s_, done))

def act(state, epsilon):
    if np.random.rand() <= epsilon:
        return random.randrange(n_actions)
    q_values = main_model.predict(state[np.newaxis], verbose=0)
    return np.argmax(q_values[0])

def replay(batch_size, gamma):
    minibatch = random.sample(memory, batch_size)
    for state, action, reward, next_state, done in minibatch:
        target = main_model.predict(state[np.newaxis], verbose=0)
        if done:
            target[0][action] = reward
        else:
            t = target_model.predict(next_state[np.newaxis], verbose=0)
            target[0][action] = reward + gamma * np.amax(t[0])
        main_model.fit(state[np.newaxis], target, epochs=1, verbose=0)

# Step 6: Train Agent
episodes = 500
batch_size = 32
gamma = 0.99
epsilon = 1.0
epsilon_min = 0.1
epsilon_decay = 0.995
scores = []

for e in range(episodes):
    obs, _ = env.reset()
    state = frame_stack.reset(obs)
    total_reward = 0
    done = False

    while not done:
        action = act(state, epsilon)
        obs_, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        next_state = frame_stack.step(obs_)
        remember(state, action, reward, next_state, done)
        state = next_state
        total_reward += reward

        if len(memory) > batch_size:
            replay(batch_size, gamma)

    if epsilon > epsilon_min:
        epsilon *= epsilon_decay

    target_model.set_weights(main_model.get_weights())
    scores.append(total_reward)
    print(f"Episode {e+1}, Score: {total_reward}, Epsilon: {epsilon:.3f}")

# Step 7: Plot Training Progress
plt.plot(scores)
plt.title("DQN Training on Breakout")
plt.xlabel("Episode")
plt.ylabel("Total Reward")
plt.grid(True)
plt.show()

env.close()


NamespaceNotFound: Namespace ALE not found. Have you installed the proper package for ALE?